# Notebook 02 — Cleaning & Preparation

**Goal:** turn the four raw exports into clean master/fact tables, with every dropped validation row logged in `data/rejects/rejected_validations.csv`.

Reads `data/interim/*_raw.parquet` (written by notebook 01). Writes `data/interim/*_clean.parquet` (+ CSV copies for easy inspection) and `data/rejects/rejected_validations.csv`.

Run the notebook from top to bottom. Do not run cells out of order.

In [17]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REJECTS_DIR = PROJECT_ROOT / "data" / "rejects"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REJECTS_DIR.mkdir(parents=True, exist_ok=True)

validations = pd.read_parquet(INTERIM_DIR / "validations_raw.parquet")
routes = pd.read_parquet(INTERIM_DIR / "routes_raw.parquet")
vehicles = pd.read_parquet(INTERIM_DIR / "vehicles_raw.parquet")
stops = pd.read_parquet(INTERIM_DIR / "stops_raw.parquet")

raw_row_count = len(validations)

print("Validations (raw):", validations.shape)
print("Routes (raw):", routes.shape)
print("Vehicles (raw):", vehicles.shape)
print("Stops (raw):", stops.shape)

Validations (raw): (81600, 10)
Routes (raw): (60, 6)
Vehicles (raw): (180, 5)
Stops (raw): (400, 5)


## 1. Remove duplicate validations

Exact duplicate rows are dropped first, then repeated `validation_id` (keeping the first occurrence). Dropping exact copies first means every removed row is counted once.

In [2]:
rejected_parts = []

exact_duplicates = validations[validations.duplicated()].copy()
exact_duplicates["reject_reason"] = "Exact duplicate row"
rejected_parts.append(exact_duplicates)

validations = validations.drop_duplicates()
print("Exact duplicate rows removed:", len(exact_duplicates))
print("Remaining:", len(validations))

duplicate_ids = validations[
    validations.duplicated(subset="validation_id", keep="first")
].copy()
duplicate_ids["reject_reason"] = "Duplicate validation_id (kept first)"
rejected_parts.append(duplicate_ids)

validations = validations.drop_duplicates(subset="validation_id", keep="first")
print("Duplicate validation_id rows removed:", len(duplicate_ids))
print("Remaining:", len(validations))

Exact duplicate rows removed: 802
Remaining: 80798
Duplicate validation_id rows removed: 798
Remaining: 80000


## 2. Parse `validation_ts`

Three timestamp formats are mixed in the raw export. Parsing them with a single generic/`"mixed"` parser silently misreads the day-first slash format as month-first for any day that is also a valid month number (`06/03/2024` becomes 3 June instead of 6 March), which misdates thousands of rows into months the data never covers (Jan–Jun 2024).

Each format is tried explicitly, in order, only on rows the previous format could not parse.

In [3]:
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",
    "%d/%m/%Y %H:%M",
    "%B %d, %Y %I:%M %p",
]


def parse_validation_timestamps(raw_ts: pd.Series) -> pd.Series:
    text = raw_ts.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=text.index, dtype="datetime64[ns]")

    for fmt in TIMESTAMP_FORMATS:
        still_missing = parsed.isna()
        parsed.loc[still_missing] = pd.to_datetime(
            text[still_missing], format=fmt, errors="coerce"
        )

    return parsed


validations["validation_ts"] = parse_validation_timestamps(validations["validation_ts"])

unparseable_ts = validations[validations["validation_ts"].isna()].copy()
unparseable_ts["reject_reason"] = "Unparseable validation_ts"
rejected_parts.append(unparseable_ts)

validations = validations[validations["validation_ts"].notna()].copy()

print("Unparseable timestamps:", len(unparseable_ts))
print("Date range:", validations["validation_ts"].min(), "to", validations["validation_ts"].max())

Unparseable timestamps: 0
Date range: 2024-01-01 00:34:50 to 2024-06-30 23:48:50


## 3. Clean ID columns

About 15% of `route_id` values had a trailing space and would not join to the routes table. Strip whitespace from every ID column, in every table.

In [4]:
validation_id_columns = ["validation_id", "route_id", "vehicle_id", "stop_id", "card_id"]
for col in validation_id_columns:
    validations[col] = validations[col].astype("string").str.strip()

routes["route_id"] = routes["route_id"].astype("string").str.strip()
vehicles["vehicle_id"] = vehicles["vehicle_id"].astype("string").str.strip()
stops["stop_id"] = stops["stop_id"].astype("string").str.strip()

print("ID columns stripped of whitespace.")

ID columns stripped of whitespace.


## 4. Normalise `fare_type`

Lower-case, strip, then map the 17 raw spellings to `Adult`, `Student`, `Senior`, `Child`, `Pass Holder`. An assertion stops the notebook if an unknown spelling appears.

In [5]:
FARE_TYPE_MAPPING = {
    "adult": "Adult",
    "full fare": "Adult",
    "student": "Student",
    "senior": "Senior",
    "senior citizen": "Senior",
    "child": "Child",
    "kid": "Child",
    "pass holder": "Pass Holder",
    "passholder": "Pass Holder",
    "monthly pass": "Pass Holder",
}

normalised_fare_type = (
    validations["fare_type"].astype("string").str.strip().str.lower()
)

unmapped_fare_types = set(normalised_fare_type.dropna()) - set(FARE_TYPE_MAPPING)
assert not unmapped_fare_types, f"Unknown fare_type spelling(s): {sorted(unmapped_fare_types)}"

validations["fare_type"] = normalised_fare_type.map(FARE_TYPE_MAPPING)

print(validations["fare_type"].value_counts(dropna=False))

fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: int64


## 5. Normalise `transfer_flag`

Six spellings of two values: `Y`/`1`/`TRUE` → `True`, `N`/`0`/`FALSE` → `False` (nullable boolean).

In [6]:
TRANSFER_FLAG_MAPPING = {
    "y": True, "1": True, "true": True,
    "n": False, "0": False, "false": False,
}

validations["transfer_flag"] = (
    validations["transfer_flag"]
    .astype("string").str.strip().str.lower()
    .map(TRANSFER_FLAG_MAPPING)
    .astype("boolean")
)

print(validations["transfer_flag"].value_counts(dropna=False))

transfer_flag
False    49573
True     30427
Name: count, dtype: Int64


## 6. Handle `passenger_count`

Null → drop and log. Negative → keep, flag `is_reversed = True` (a reversed validation cancels its original boarding, so boardings and revenue end up net).

In [7]:
missing_passenger = validations[validations["passenger_count"].isna()].copy()
missing_passenger["reject_reason"] = "Missing passenger_count"
rejected_parts.append(missing_passenger)

validations = validations[validations["passenger_count"].notna()].copy()
validations["passenger_count"] = validations["passenger_count"].astype(int)
validations["is_reversed"] = validations["passenger_count"] < 0

print("Missing passenger_count rows removed:", len(missing_passenger))
print("Reversed (negative) validations kept:", int(validations["is_reversed"].sum()))
print("Remaining:", len(validations))

Missing passenger_count rows removed: 811
Reversed (negative) validations kept: 358
Remaining: 79189


## 7. Handle `fare_amount`

Null → median of the same `fare_type`. Above the 99.5th percentile → capped at that percentile, calculated per fare type (fares depend on ticket type, so one overall median or cap would be wrong for most rows).

In [8]:
validations["fare_amount"] = validations.groupby("fare_type")["fare_amount"].transform(
    lambda s: s.fillna(s.median())
)

fare_caps = validations.groupby("fare_type")["fare_amount"].transform(
    lambda s: s.quantile(0.995)
)
validations["fare_amount"] = validations["fare_amount"].clip(upper=fare_caps)

print(validations.groupby("fare_type")["fare_amount"].describe())

               count      mean       std   min   25%  50%  75%     max
fare_type                                                             
Adult        43930.0  2.500409  0.147984  1.88  2.40  2.5  2.6  2.8900
Child         6263.0  0.996789  0.149956  0.32  0.89  1.0  1.1  1.3969
Pass Holder   7883.0  0.057978  0.085479  0.00  0.00  0.0  0.1  0.3800
Senior        9376.0  1.196882  0.148573  0.60  1.10  1.2  1.3  1.6000
Student      11737.0  1.497546  0.148209  0.95  1.40  1.5  1.6  1.8800


## 8. Handle missing `card_id`

Blank → `SINGLE_RIDE`. Paper tickets have no card; they are valid boardings but are left out of `unique_cards`.

In [9]:
validations["card_id"] = (
    validations["card_id"]
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    .fillna("SINGLE_RIDE")
)

print("SINGLE_RIDE validations:", (validations["card_id"] == "SINGLE_RIDE").sum())

SINGLE_RIDE validations: 9448


## 9. Clean the master tables

- **routes**: `route_name` trimmed + title case; `mode` trimmed + title case (10 spellings → Bus/Tram/Metro); null `district` → `Unknown`; null `route_length_km` → median of the same mode.
- **vehicles**: `year_manufactured` outside 1980–2026 → null; `vehicle_age_band` from reference year 2026: `0-5`, `6-10`, `11-20`, `20+`, `Unknown`; null `capacity` → median of the same `vehicle_type`.
- **stops**: `stop_name` trimmed + title case; `has_shelter` (`Y`/`N`/`1`/`0`) → boolean.

In [10]:
routes["route_name"] = routes["route_name"].astype("string").str.strip().str.title()
routes["mode"] = routes["mode"].astype("string").str.strip().str.title()
routes["district"] = routes["district"].astype("string").str.strip().fillna("Unknown")
routes["route_length_km"] = routes.groupby("mode")["route_length_km"].transform(
    lambda s: s.fillna(s.median())
)

print(routes[["route_name", "mode", "district", "route_length_km"]].isna().sum())

route_name         0
mode               0
district           0
route_length_km    0
dtype: int64


In [11]:
REFERENCE_YEAR = 2026

vehicles.loc[
    ~vehicles["year_manufactured"].between(1980, REFERENCE_YEAR),
    "year_manufactured",
] = pd.NA

vehicle_age = REFERENCE_YEAR - vehicles["year_manufactured"]
vehicles["vehicle_age_band"] = (
    pd.cut(vehicle_age, bins=[-1, 5, 10, 20, float("inf")], labels=["0-5", "6-10", "11-20", "20+"])
    .astype("string")
    .fillna("Unknown")
)

vehicles["capacity"] = vehicles.groupby("vehicle_type")["capacity"].transform(
    lambda s: s.fillna(s.median())
)

print(vehicles["vehicle_age_band"].value_counts(dropna=False))
print("Missing capacity after imputation:", vehicles["capacity"].isna().sum())
vehicles["capacity"] = vehicles["capacity"].round().astype(int)


vehicle_age_band
20+        84
11-20      44
6-10       23
Unknown    15
0-5        14
Name: count, dtype: int64[pyarrow]
Missing capacity after imputation: 0


In [12]:
SHELTER_MAPPING = {"y": True, "1": True, "true": True, "n": False, "0": False, "false": False}

stops["stop_name"] = stops["stop_name"].astype("string").str.strip().str.title()
stops["has_shelter"] = (
    stops["has_shelter"]
    .astype("string").str.strip().str.lower()
    .map(SHELTER_MAPPING)
    .astype("boolean")
)

print(stops["has_shelter"].value_counts(dropna=False))

has_shelter
False    211
True     189
Name: count, dtype: Int64


## 10. Referential integrity

Validations whose `route_id`, `vehicle_id` or `stop_id` is not in the cleaned master tables are rejected. `card_id` is not checked, so `SINGLE_RIDE` validations are kept.

In [13]:
valid_route_ids = set(routes["route_id"].dropna())
valid_vehicle_ids = set(vehicles["vehicle_id"].dropna())
valid_stop_ids = set(stops["stop_id"].dropna())

invalid_reference = (
    ~validations["route_id"].isin(valid_route_ids)
    | ~validations["vehicle_id"].isin(valid_vehicle_ids)
    | ~validations["stop_id"].isin(valid_stop_ids)
)

invalid_references = validations[invalid_reference].copy()
invalid_references["reject_reason"] = "Unknown route_id, vehicle_id, or stop_id"
rejected_parts.append(invalid_references)

validations = validations[~invalid_reference].copy()

print("Rows with an unknown route/vehicle/stop id:", len(invalid_references))
print("Remaining:", len(validations))

Rows with an unknown route/vehicle/stop id: 40
Remaining: 79149


## 11. Derived columns

`date`, `year`, `month`, `year_month` (YYYY-MM), `quarter`, `hour`, `day_of_week`, `is_weekend`, `time_period` (`Night (00-03)`, `Early (04-06)`, `AM Peak (07-09)`, `Midday (10-15)`, `PM Peak (16-18)`, `Evening (19-23)`) and `revenue = fare_amount × passenger_count`.

In [14]:
validations["date"] = validations["validation_ts"].dt.date
validations["year"] = validations["validation_ts"].dt.year
validations["month"] = validations["validation_ts"].dt.month
validations["year_month"] = validations["validation_ts"].dt.to_period("M").astype(str)
validations["quarter"] = validations["validation_ts"].dt.quarter
validations["hour"] = validations["validation_ts"].dt.hour
validations["day_of_week"] = validations["validation_ts"].dt.day_name()
validations["is_weekend"] = validations["validation_ts"].dt.day_of_week >= 5

validations["time_period"] = pd.cut(
    validations["hour"],
    bins=[-1, 3, 6, 9, 15, 18, 23],
    labels=[
        "Night (00-03)",
        "Early (04-06)",
        "AM Peak (07-09)",
        "Midday (10-15)",
        "PM Peak (16-18)",
        "Evening (19-23)",
    ],
).astype("string")

validations["revenue"] = (validations["fare_amount"] * validations["passenger_count"]).round(2)

print(validations[["date", "year_month", "hour", "time_period", "revenue"]].head())

         date year_month  hour      time_period  revenue
0  2024-05-13    2024-05    18  PM Peak (16-18)     2.53
1  2024-06-06    2024-06     8  AM Peak (07-09)     2.48
2  2024-03-06    2024-03    13   Midday (10-15)     2.85
3  2024-04-21    2024-04    17  PM Peak (16-18)     4.92
4  2024-06-14    2024-06    18  PM Peak (16-18)     1.34


## 12. Save the rejected rows and assert the row count

`raw rows − rejected rows` must equal the number of clean rows that survive.

In [15]:
all_rejected = pd.concat(rejected_parts, ignore_index=True)

all_rejected.to_csv(REJECTS_DIR / "rejected_validations.csv", index=False)

print(all_rejected["reject_reason"].value_counts())
print("\nTotal rejected rows:", len(all_rejected))

assert raw_row_count - len(all_rejected) == len(validations), (
    "raw - rejected does not equal the number of clean rows."
)
print("\nRow-count check passed:", raw_row_count, "-", len(all_rejected), "=", len(validations))

reject_reason
Missing passenger_count                     811
Exact duplicate row                         802
Duplicate validation_id (kept first)        798
Unknown route_id, vehicle_id, or stop_id     40
Name: count, dtype: int64

Total rejected rows: 2451

Row-count check passed: 81600 - 2451 = 79149


## 13. Save the cleaned tables

Saved as Parquet (keeps dtypes between notebooks) with CSV copies for easy inspection.

In [16]:
validations.to_parquet(INTERIM_DIR / "validations_clean.parquet", index=False)
routes.to_parquet(INTERIM_DIR / "routes_clean.parquet", index=False)
vehicles.to_parquet(INTERIM_DIR / "vehicles_clean.parquet", index=False)
stops.to_parquet(INTERIM_DIR / "stops_clean.parquet", index=False)

validations.to_csv(INTERIM_DIR / "validations_clean.csv", index=False)
routes.to_csv(INTERIM_DIR / "routes_clean.csv", index=False)
vehicles.to_csv(INTERIM_DIR / "vehicles_clean.csv", index=False)
stops.to_csv(INTERIM_DIR / "stops_clean.csv", index=False)

print("Cleaned tables saved.")
print("Validations:", validations.shape)
print("Routes:", routes.shape)
print("Vehicles:", vehicles.shape)
print("Stops:", stops.shape)

Cleaned tables saved.
Validations: (79149, 21)
Routes: (60, 6)
Vehicles: (180, 6)
Stops: (400, 5)
